In [1]:
import copy, subprocess, re
from pathlib import Path
import pandas as pd

SBD = Path.home() / "sqd-project/sbd/apps/chemistry_tpb_selected_basis_diagonalization/diag"
OUT = Path.home() / "sqd-project/outputs"

E_HF, E_CASCI = -108.8677463470, -108.9467159424

def run_sbd(adet, bdet, fcidump=OUT / "n2_cas66.fcidump"):
    """Run sbd and parse the ground-state energy."""
    r = subprocess.run(
        ["mpirun", "-np", "1", str(SBD),
         "--fcidump", str(fcidump),
         "--adetfile", str(adet), "--bdetfile", str(bdet),
         "--method", "0", "--iteration", "20",
         "--block", "100", "--tolerance", "1e-8"],
        capture_output=True, text=True, cwd=SBD.parent)
    m = re.search(r"Sample-based diagonalization: Energy = ([-\d.]+)", r.stdout)
    if not m:
        raise RuntimeError(r.stdout[-2000:] + r.stderr[-2000:])
    return float(m.group(1))

rows = []
for label, bitfile in [("masked",   "n2_cas66_bitstrings.txt"),
                       ("unmasked", "n2_cas66_bitstrings_unmasked.txt")]:
    for budget in [3, 5, 8, 10, 15, None]:
        cfg = copy.deepcopy(CONFIG)
        cfg["bitstring_filename"] = bitfile
        cfg["max_determinants"]   = budget
        res = stage2b.run_stage2b(cfg)

        # adjust these two lines to match what run_stage2b actually returns
        adet, bdet = Path(res.adet_path), Path(res.bdet_path)

        n_a = len(adet.read_text().strip().split("\n"))
        n_b = len(bdet.read_text().strip().split("\n"))
        E   = run_sbd(adet, bdet)

        rows.append({
            "ansatz": label,
            "budget": budget if budget else "all",
            "n_alpha": n_a, "n_beta": n_b,
            "dim": n_a * n_b,
            "pct_CI": 100 * n_a * n_b / 400,
            "E": E,
            "err_mHa": (E - E_CASCI) * 1000,
            "pct_corr": 100 * (E - E_HF) / (E_CASCI - E_HF),
        })

df = pd.DataFrame(rows)
print(df.round(4).to_string(index=False))

NameError: name 'CONFIG' is not defined